In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mabilisang pagsisimula sa Executor

<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekumenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Katulad ng [Sampler](/guides/get-started-with-sampler) primitive, ang Executor ay nag-sa-sample ng mga output register mula sa mga quantum circuit execution, ngunit wala itong built-in na error suppression o mitigation. Sa halip, ito ay bahagi ng [directed execution model](/guides/directed-execution-model) na nagbibigay ng mga sangkap para makuha ang mga design intent sa client side, at inililipat ang mahal na paglikha ng mga variant ng circuit sa server side. Sinusunod ng Executor ang mga direktibo na ibinigay sa mga annotation at opsyon ng circuit, nag-ge-generate at nag-ba-bind ng mga halaga ng parameter, nagpapatupad ng mga bound circuit sa hardware, at ibinabalik ang mga resulta ng execution at metadata. Hindi ito gumagawa ng anumang implicit na desisyon para sa iyo at nagbibigay ng buong kontrol at transparency.

> **Note:** Ang Qiskit package ay wala pang base class para sa Executor primitive.

## Bago ka magsimula
Ang ilang halimbawa ng code sa pahinang ito ay gumagamit ng `samplex`, na bahagi ng Samplomatic package. Samakatuwid, bago patakbuhin ang mga code block na iyon, kailangan mong i-install ang Samplomatic, tulad ng ipinapakita sa sumusunod na code block. Para sa karagdagang impormasyon, tingnan ang [dokumentasyon ng Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Lumikha at mag-transpile ng circuit
Kailangan mo ng kahit isang circuit para gamitin ang Executor primitive. Maaari itong may mga opsyonal na parameter.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Ang circuit ay kailangang i-transform para gumamit lamang ng mga tagubilin na sinusuportahan ng QPU (tinatawag na *instruction set architecture (ISA)* circuit). Gamitin ang transpiler para gawin ito.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. I-initialize ang isang `QuantumProgram`
I-initialize ang isang `QuantumProgram` kasama ang iyong workload. Ang isang `QuantumProgram` ay binubuo ng mga `QuantumProgramItem`. Karaniwang, ang bawat item ay binubuo ng isang circuit, isang set ng mga halaga ng parameter, at posibleng isang `samplex` para gawing random ang nilalaman ng circuit. Para sa buong detalye, tingnan ang [Mga input at output ng Executor](/guides/executor-input-output).

Ang sumusunod na cell ay nag-i-initialize ng isang `QuantumProgram` at nagtatakda na magsagawa ng 25 shot. Susunod, ini-append nito ang transpiled na target na circuit.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opsyonal: Grupuhin ang mga gate at measurement sa mga annotated na box
Ang pag-grupo ng mga tagubilin sa mga box at pag-annotate sa mga ito ay ang pangunahing paraan para tukuyin ang iyong intent. Sa sumusunod na halimbawa, ginagamit namin ang `generate_boxing_pass_manager` at ang mga parameter ng twirling nito para grupuhin ang mga two-qubit gate at mga measurement sa mga box at mag-apply ng twirling annotation.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. I-invoke ang Executor at makuha ang mga resulta
Patakbuhin ang `QuantumProgram` sa isang IBM&reg; backend sa pamamagitan ng paggamit ng `Executor` primitive na may mga default na opsyon. Tingnan ang [Mga opsyon ng Executor](/guides/executor-options) para malaman ang tungkol sa mga available na opsyon.